# Evaluating HGLM's segmentation objective

Does HGLM's tailoring process identify the max F1 region?

Approach:
- sample `n` effect extents
- for each, impose effect at multiple `f_ratio`
- segment according to HGLM
- choose the max F1 score, declare it and all of its ancestors & descendents as significant
- run the LL tailoring process, compare the f1 score of its output to the f1 score of the target region

todo: add sens & spec too

In [ ]:
import hglm
import numpy as np

n_jobs = -1
n_repeat = 20
radius = 4
effect_perc = .2
f_ratio = np.logspace(np.log10(.001), np.log10(.05), 9)

# # load human connectome project data
# folder = '/home/matt/Dropbox/pnl_hglm/data/hcp100_lowres/image'
# exp_hcp = hglm.experiment.ExperimentImageOnly.from_search(
#     folder=folder,
#     sbj_regex=r'[\d]{6}',
#     img_glob_dict={'FA': '*_FA.nii.gz',
#                    'MD': '*_MD.nii.gz'})    
# exp = exp_hcp.sample_x(a=2, seed=0, add_bias=True)

# wgn
exp = hglm.experiment.Experiment.from_gauss(seed=0,
                                            shape=(5, 5, 5),
                                            a=2,
                                            b=2,
                                            num_img=100)

In [ ]:
from joblib import Parallel, delayed
from tqdm import tqdm
import pandas as pd
from itertools import product

def process_seed_f(seed, f, radius, exp, effect_perc):
    # trim experiment
    extenter = hglm.effect.ExtenterSphere(radius=radius)
    mask = extenter(mask_idx=exp.mask_idx, seed=seed, contiguous=True)
    exp_masked = exp.apply_mask(mask)

    # sample extent
    n = exp_masked.y.shape[2] * effect_perc
    extenter = hglm.effect.ExtenterMinVar(n=n)
    mask_target = extenter(y=exp_masked.y,
                           mask_idx=exp_masked.mask_idx,
                           seed=seed)

    # impose effect
    _exp, effect = exp_masked.impose_effect(mask=mask_target,
                                            seed=seed, f_ratio=f)
    
    # segment
    children = hglm.experiment.AnalysisHGLM.cluster(_exp, mode='full')
    f1 = hglm.graph.get_f1(mask=mask_target,
                           mask_idx=_exp.mask_idx,
                           children=children)
    
    # "hypothesis testing": declare target region & all ancestors/descendents as significant
    reg_target = f1.argmax()
    num_leaf = _exp.y.shape[2]
    sig_reg_list = list(hglm.graph.iter_topo(children=children, 
                                             num_leaf=num_leaf, 
                                             node_start=reg_target))
    assert reg_target in sig_reg_list
    
    node = reg_target
    parent = hglm.graph.get_parent(children, num_leaf)
    while parent[node] != -1:
        p = parent[node]
        sig_reg_list.append(p)
        node = p
        
    reg_out_list = hglm.experiment.AnalysisHGLM.tailor(sig_reg_list=sig_reg_list,
                                                       children=children,
                                                       exp=_exp,
                                                       alpha_tailor=.05,
                                                       n_perm=100)
        
    return dict(seed=seed, 
                f_ratio=f, 
                f1_max=f1.max(), 
                f1_max_out=f1[reg_out_list].max())

param_grid = list(product(range(int(n_repeat)), f_ratio))

results = Parallel(n_jobs=n_jobs)(
    delayed(process_seed_f)(seed, f, radius, exp, effect_perc)
    for seed, f in tqdm(param_grid, desc='experiment (per seed-stat)')
)


df = pd.DataFrame(results)

In [ ]:
df['f1_max_ratio'] = df['f1_max_out'] / df['f1_max']

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --- Feature Parameters ---
x_feature = 'f_ratio'
y_feature = 'f1_max_ratio'
group_feature = 'seed'

# --- Plotting ---
plt.figure(figsize=(10, 6))

# Plot each seed line (black, no marker, no legend)
for _, group_df in df.groupby(group_feature):
    group_df = group_df.sort_values(by=x_feature)
    plt.plot(group_df[x_feature], group_df[y_feature],
             color='grey', linewidth=1)

# Plot bold average line (black, with legend)
avg_df = df.groupby(x_feature, as_index=False)[y_feature].mean()
avg_df = avg_df.sort_values(by=x_feature)

plt.plot(avg_df[x_feature], avg_df[y_feature],
         color='black', linewidth=5, label='Average')

# Axis formatting
plt.xscale('log')
plt.xlabel(x_feature)
plt.ylabel(y_feature)
plt.title(f'{y_feature} vs {x_feature} (Averaged Across Seeds)')
plt.legend(title='', loc='best')
plt.tight_layout()
plt.show()
